# Finance in AI Credit Risk Analysis

This notebook analyzes the Home Credit dataset using the chapter structure you provided.

The goal is not just to show charts. The goal is to justify why each analysis block belongs in a serious credit-risk review: what part of the lending process it represents, what business question it answers, and why the result is relevant for underwriting, monitoring, or compliance.

- credit lifecycle and domain layers
- willingness vs capacity to repay
- time-based performance through vintage and survival analysis
- core credit metrics: AUC, KS, PSI
- binning, WoE, IV, and special-value handling
- practical mapping to the 4-layer framework

The structure is intentionally domain-first. In credit risk, a model is only useful if it can be explained as part of a lending system, not as an isolated machine learning score.

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "dataset").exists() and (ROOT.parent / "dataset").exists():
    ROOT = ROOT.parent
if not (ROOT / "dataset").exists():
    raise RuntimeError("Run this notebook from the repo root or from notebooks/.")

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

DATASET_DIR = ROOT / "dataset"
OUTPUTS_DIR = ROOT / "outputs"
ROOT

In [ ]:
import json
import re
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.metrics import roc_auc_score

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 180)
plt.style.use("ggplot")

def safe_divide(a, b):
    if isinstance(b, pd.Series):
        b = b.replace({0: np.nan})
    return a / b

def ks_statistic(y_true, y_score):
    order = np.argsort(y_score)
    y_true = np.asarray(y_true)[order]
    bad = np.cumsum(y_true == 1) / max((y_true == 1).sum(), 1)
    good = np.cumsum(y_true == 0) / max((y_true == 0).sum(), 1)
    return float(np.max(np.abs(bad - good)))

def psi(expected, actual, bins=10):
    expected = np.asarray(expected, dtype=float)
    actual = np.asarray(actual, dtype=float)
    expected = expected[np.isfinite(expected)]
    actual = actual[np.isfinite(actual)]
    quantiles = np.unique(np.quantile(expected, np.linspace(0, 1, bins + 1)))
    if quantiles.size < 3:
        return 0.0
    expected_hist, _ = np.histogram(expected, bins=quantiles)
    actual_hist, _ = np.histogram(actual, bins=quantiles)
    expected_pct = np.clip(expected_hist / max(expected_hist.sum(), 1), 1e-6, None)
    actual_pct = np.clip(actual_hist / max(actual_hist.sum(), 1), 1e-6, None)
    return float(np.sum((actual_pct - expected_pct) * np.log(actual_pct / expected_pct)))

def risk_ladder(df, feature, bins=5):
    sample = df[[feature, "TARGET"]].dropna().copy()
    sample["bucket"] = pd.qcut(sample[feature], q=bins, duplicates="drop")
    grouped = (
        sample.groupby("bucket", observed=True)["TARGET"]
        .agg(default_rate="mean", count="size")
        .reset_index()
    )
    grouped["feature"] = feature
    grouped["bucket"] = grouped["bucket"].astype(str)
    return grouped[["feature", "bucket", "count", "default_rate"]]

def woe_iv_table(series, target, bins=5):
    frame = pd.DataFrame({"x": series, "y": target}).copy()
    missing_mask = frame["x"].isna()
    valid = frame.loc[~missing_mask].copy()
    valid["bucket"] = pd.qcut(valid["x"], q=bins, duplicates="drop")
    frame.loc[~missing_mask, "bucket"] = valid["bucket"].astype(str).values
    frame.loc[missing_mask, "bucket"] = "<MISSING>"
    grouped = frame.groupby("bucket", observed=True)["y"].agg(bad_count="sum", count="size").reset_index()
    grouped["good_count"] = grouped["count"] - grouped["bad_count"]
    total_bad = grouped["bad_count"].sum()
    total_good = grouped["good_count"].sum()
    grouped["bad_rate"] = grouped["bad_count"] / max(total_bad, 1)
    grouped["good_rate"] = grouped["good_count"] / max(total_good, 1)
    grouped["woe"] = np.log((grouped["bad_rate"] + 1e-9) / (grouped["good_rate"] + 1e-9))
    grouped["iv_bin"] = (grouped["bad_rate"] - grouped["good_rate"]) * grouped["woe"]
    iv = grouped["iv_bin"].sum()
    return grouped.sort_values("bucket"), float(iv)

def kaplan_meier_curve(durations, events, max_time=12):
    durations = np.asarray(durations, dtype=int)
    events = np.asarray(events, dtype=int)
    rows = []
    survival = 1.0
    for t in range(1, max_time + 1):
        at_risk = int((durations >= t).sum())
        defaults = int(((durations == t) & (events == 1)).sum())
        if at_risk > 0:
            survival *= (1 - defaults / at_risk)
        rows.append({"month_on_book": t, "at_risk": at_risk, "defaults": defaults, "survival_probability": survival})
    return pd.DataFrame(rows)

In [ ]:
application_train = pd.read_csv(DATASET_DIR / "application_train.csv")
application_test = pd.read_csv(DATASET_DIR / "application_test.csv")
previous = pd.read_csv(
    DATASET_DIR / "previous_application.csv",
    usecols=["SK_ID_PREV", "SK_ID_CURR", "NAME_CONTRACT_STATUS", "NAME_CONTRACT_TYPE", "AMT_CREDIT", "AMT_ANNUITY", "CNT_PAYMENT", "DAYS_DECISION"],
)
installments = pd.read_csv(
    DATASET_DIR / "installments_payments.csv",
    usecols=["SK_ID_PREV", "SK_ID_CURR", "NUM_INSTALMENT_NUMBER", "DAYS_INSTALMENT", "DAYS_ENTRY_PAYMENT", "AMT_INSTALMENT", "AMT_PAYMENT"],
)
bureau = pd.read_csv(
    DATASET_DIR / "bureau.csv",
    usecols=["SK_ID_CURR", "CREDIT_ACTIVE", "DAYS_CREDIT", "CREDIT_DAY_OVERDUE", "AMT_CREDIT_SUM", "AMT_CREDIT_SUM_DEBT", "AMT_CREDIT_SUM_OVERDUE"],
)

master_path = OUTPUTS_DIR / "features" / "master_table.pkl"
master = pd.read_pickle(master_path) if master_path.exists() else None

print("application_train:", application_train.shape)
print("application_test:", application_test.shape)
print("previous_application:", previous.shape)
print("installments_payments:", installments.shape)
print("bureau:", bureau.shape)
print("master_table available:", master is not None)

## 4.1 Credit Lifecycle and Domain Mapping

This section establishes why the dataset should be read as a credit lifecycle dataset rather than a flat binary classification table.

Justification:
- A defensible credit analysis must separate origination evidence, in-loan behavior, and post-loan outcomes.
- That separation matters because feature design, model choice, monitoring controls, and compliance expectations differ by phase.
- In Home Credit, the target sits at the application level, but the real risk evidence is distributed across bureau, previous applications, and repayment history. Mapping those tables to lifecycle phases prevents the analysis from becoming a random feature inventory.

In [ ]:
lifecycle_mapping = pd.DataFrame(
    [
        {"phase": "Pre-loan", "table": "application_train/application_test", "role": "origination, borrower profile, initial affordability and eligibility"},
        {"phase": "Pre-loan", "table": "bureau", "role": "external willingness and indebtedness evidence"},
        {"phase": "Pre-loan", "table": "previous_application", "role": "past product demand, approval/refusal history, prior terms"},
        {"phase": "In-loan", "table": "installments_payments", "role": "repayment trajectory, delinquency emergence, cure behavior"},
        {"phase": "In-loan", "table": "POS_CASH_balance / credit_card_balance", "role": "line usage, utilization, arrears, revolving stress"},
        {"phase": "Post-loan", "table": "TARGET + delinquency proxies", "role": "default outcome, recovery proxy, model feedback loop"},
    ]
)
lifecycle_mapping

In [ ]:
def enrich_application(df):
    frame = df.copy()
    frame["DAYS_EMPLOYED_ANOM"] = (frame["DAYS_EMPLOYED"] == 365243).astype(int)
    frame.loc[frame["DAYS_EMPLOYED"] == 365243, "DAYS_EMPLOYED"] = np.nan
    ext = frame[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    frame["payment_rate"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_CREDIT"])
    frame["credit_income_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_INCOME_TOTAL"])
    frame["annuity_income_pct"] = safe_divide(frame["AMT_ANNUITY"], frame["AMT_INCOME_TOTAL"])
    frame["credit_annuity_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_ANNUITY"])
    frame["credit_goods_ratio"] = safe_divide(frame["AMT_CREDIT"], frame["AMT_GOODS_PRICE"])
    frame["income_per_person"] = safe_divide(frame["AMT_INCOME_TOTAL"], frame["CNT_FAM_MEMBERS"])
    frame["age_years"] = -frame["DAYS_BIRTH"] / 365.0
    frame["employment_years"] = -frame["DAYS_EMPLOYED"] / 365.0
    frame["ext_source_mean"] = ext.mean(axis=1)
    frame["ext_source_min"] = ext.min(axis=1)
    frame["ext_source_max"] = ext.max(axis=1)
    frame["ext_source_count"] = ext.notna().sum(axis=1)
    return frame

train = enrich_application(application_train)
test = enrich_application(application_test)

portfolio_snapshot = pd.DataFrame(
    [
        {
            "dataset": "train",
            "rows": len(train),
            "customers": train["SK_ID_CURR"].nunique(),
            "default_rate": train["TARGET"].mean(),
            "avg_income": train["AMT_INCOME_TOTAL"].mean(),
            "avg_credit": train["AMT_CREDIT"].mean(),
            "avg_annuity": train["AMT_ANNUITY"].mean(),
        },
        {
            "dataset": "test",
            "rows": len(test),
            "customers": test["SK_ID_CURR"].nunique(),
            "default_rate": np.nan,
            "avg_income": test["AMT_INCOME_TOTAL"].mean(),
            "avg_credit": test["AMT_CREDIT"].mean(),
            "avg_annuity": test["AMT_ANNUITY"].mean(),
        },
    ]
)
portfolio_snapshot

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
train["TARGET"].value_counts().sort_index().plot(kind="bar", ax=axes[0], color=["#4C78A8", "#E45756"])
axes[0].set_title("Target Distribution")
axes[0].set_xlabel("TARGET")
axes[0].set_ylabel("Count")

portfolio_snapshot.set_index("dataset")[["avg_income", "avg_credit", "avg_annuity"]].plot(kind="bar", ax=axes[1])
axes[1].set_title("Train vs Test Portfolio Profile")
axes[1].tick_params(axis="x", rotation=0)
axes[1].set_ylabel("Amount")
plt.tight_layout()
plt.show()

## 4.1.2 The Two Pillars: Capacity and Willingness

This section follows the core credit-risk idea that repayment depends on two different mechanisms: the ability to pay and the intent or discipline to pay.

Justification:
- Capacity proxies capture affordability and cash-flow stress: income, annuity burden, credit burden, and household dilution.
- Willingness proxies capture behavioral and external repayment signals: external scores, bureau stress, past refusal history, and delinquency behavior.
- Keeping these pillars separate makes the analysis easier to explain. A borrower can show financial capacity but weak repayment discipline, or strong willingness but limited affordability. That distinction is useful for underwriting, pricing, and adverse-action reasoning.

In [ ]:
if master is not None:
    master_train = master[master["TARGET"].notna()].copy()
    master_train["TARGET"] = master_train["TARGET"].astype(int)
    master_train = enrich_application(master_train)
else:
    master_train = train.copy()

capacity_features = [
    feature for feature in [
        "payment_rate",
        "credit_income_ratio",
        "annuity_income_pct",
        "credit_annuity_ratio",
        "income_per_person",
    ]
    if feature in master_train.columns
]
willingness_features = [
    feature for feature in [
        "ext_source_mean",
        "bureau_total_debt_ratio",
        "install_late_payment_rate",
        "previous_refused_count",
    ]
    if feature in master_train.columns
]

capacity_ladders = pd.concat([risk_ladder(master_train, feature) for feature in capacity_features], ignore_index=True)
willingness_ladders = pd.concat([risk_ladder(master_train, feature) for feature in willingness_features], ignore_index=True)

display(capacity_ladders)
display(willingness_ladders)

In [ ]:
max_cols = max(1, len(capacity_features), len(willingness_features))
fig, axes = plt.subplots(2, max_cols, figsize=(4 * max_cols, 8), squeeze=False)
for i, feature in enumerate(capacity_features):
    plot_df = capacity_ladders[capacity_ladders["feature"] == feature]
    axes[0, i].plot(range(len(plot_df)), plot_df["default_rate"], marker="o", color="#E45756")
    axes[0, i].set_title(f"Capacity: {feature}")
    axes[0, i].set_xticks(range(len(plot_df)))
    axes[0, i].set_xticklabels(plot_df["bucket"], rotation=45, ha="right")
    axes[0, i].set_ylabel("Default Rate")
for i in range(len(capacity_features), max_cols):
    axes[0, i].axis("off")
for i, feature in enumerate(willingness_features):
    plot_df = willingness_ladders[willingness_ladders["feature"] == feature]
    axes[1, i].plot(range(len(plot_df)), plot_df["default_rate"], marker="o", color="#4C78A8")
    axes[1, i].set_title(f"Willingness: {feature}")
    axes[1, i].set_xticks(range(len(plot_df)))
    axes[1, i].set_xticklabels(plot_df["bucket"], rotation=45, ha="right")
    axes[1, i].set_ylabel("Default Rate")
for i in range(len(willingness_features), max_cols):
    axes[1, i].axis("off")
plt.tight_layout()
plt.show()

## 4.2.2 Time-Based Performance: Vintage Analysis and Survival

This section treats default as a path over time, not as a single label attached to the customer forever.

Justification:
- A binary default target tells us who eventually failed, but not when deterioration began.
- Vintage analysis shows whether newer cohorts season worse than earlier ones at the same point in their life.
- Survival analysis shows how long accounts stay current before a serious delinquency event emerges.
- In this dataset, dates are anonymized and relative, so the cohorting is an approximation based on `DAYS_DECISION` and installment sequence. That limitation should be stated clearly when presenting the charts: these are portfolio-timing diagnostics, not audited accounting vintages.

In [ ]:
installments_time = installments.copy()
installments_time["days_past_due"] = (installments_time["DAYS_ENTRY_PAYMENT"] - installments_time["DAYS_INSTALMENT"]).clip(lower=0)
severe_dpd = installments_time[installments_time["days_past_due"] >= 30].groupby("SK_ID_PREV")["NUM_INSTALMENT_NUMBER"].min().rename("first_severe_dpd_mob")
loan_horizon = installments_time.groupby("SK_ID_PREV")["NUM_INSTALMENT_NUMBER"].max().rename("last_observed_mob")
loan_perf = pd.concat([loan_horizon, severe_dpd], axis=1).reset_index()
loan_perf["event_observed"] = loan_perf["first_severe_dpd_mob"].notna().astype(int)
loan_perf["time_to_event"] = loan_perf["first_severe_dpd_mob"].fillna(loan_perf["last_observed_mob"]).astype(int)

approved_prev = previous[previous["NAME_CONTRACT_STATUS"] == "Approved"].copy()
approved_prev["decision_month"] = (-approved_prev["DAYS_DECISION"] // 30).astype(int)
approved_prev = approved_prev[approved_prev["decision_month"].between(1, 12)]
approved_prev = approved_prev.merge(loan_perf, on="SK_ID_PREV", how="inner")

vintage_rows = []
for cohort in sorted(approved_prev["decision_month"].unique())[:8]:
    cohort_df = approved_prev[approved_prev["decision_month"] == cohort]
    if len(cohort_df) < 200:
        continue
    for mob in range(1, 13):
        cumulative_default = ((cohort_df["event_observed"] == 1) & (cohort_df["time_to_event"] <= mob)).mean()
        vintage_rows.append({"cohort": f"M-{cohort}", "month_on_book": mob, "cumulative_default_rate": cumulative_default, "loan_count": len(cohort_df)})

vintage_df = pd.DataFrame(vintage_rows)
vintage_df.head(20)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for cohort, cohort_df in vintage_df.groupby("cohort", observed=True):
    ax.plot(cohort_df["month_on_book"], cohort_df["cumulative_default_rate"], marker="o", label=cohort)
ax.set_title("Vintage Analysis: Severe 30+ DPD by Relative Origination Cohort")
ax.set_xlabel("Months on Book")
ax.set_ylabel("Cumulative Severe Delinquency Rate")
ax.legend(title="Cohort", loc="upper left")
plt.tight_layout()
plt.show()

In [ ]:
survival_input = approved_prev.copy()
survival_input["duration_12m"] = survival_input["time_to_event"].clip(upper=12)
survival_input["event_12m"] = ((survival_input["event_observed"] == 1) & (survival_input["time_to_event"] <= 12)).astype(int)
km = kaplan_meier_curve(survival_input["duration_12m"], survival_input["event_12m"], max_time=12)
km

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))
ax.step(km["month_on_book"], km["survival_probability"], where="post", color="#4C78A8")
ax.set_title("Kaplan-Meier Style Survival Curve: Staying Current Through 12 MOB")
ax.set_xlabel("Months on Book")
ax.set_ylabel("Survival Probability")
plt.tight_layout()
plt.show()

## 4.3 Domain-Driven 4-Layer Framework Mapped to Home Credit

This framework converts the notebook from an exploratory exercise into an operating view of credit risk.

Justification:
- The Data Assets layer asks whether the institution has the right borrower, bureau, and behavioral signals under proper governance.
- The Model layer asks how probability of default or delinquency is estimated.
- The Strategy and Monitoring layer asks how a score becomes a lending action and how drift is detected over time.
- The Application layer asks where the decision lands in the workflow: origination, line management, or collections.
- This matters because a high model score on its own is not a credit system. The score must be connected to policy, monitoring, and operational action.

In [ ]:
layer_mapping = pd.DataFrame(
    [
        {
            "layer": "Data Assets",
            "home_credit_mapping": "application, bureau, previous_application, installments, POS, credit_card tables",
            "why_it_matters": "raw profile, bureau, behavioral, and payment data for both origination and monitoring",
        },
        {
            "layer": "Model",
            "home_credit_mapping": "scorecards, booster models, survival-style delinquency models, reason-code features",
            "why_it_matters": "predict PD, stage delinquency risk, and identify the strongest risk drivers",
        },
        {
            "layer": "Strategy & Monitoring",
            "home_credit_mapping": "accept/reject cutoffs, limit/pricing rules, PSI drift checks, vintage monitoring, fairness review",
            "why_it_matters": "translates raw risk scores into credit policy and keeps the system stable over time",
        },
        {
            "layer": "Application",
            "home_credit_mapping": "origination decisions, in-loan early warning, collections prioritization, adverse-action explanation",
            "why_it_matters": "where the model becomes an operational lending action",
        },
    ]
)
layer_mapping

## 4.4 Essential Metrics and Techniques

This section keeps the notebook aligned with credit practice rather than generic machine learning reporting.

Justification:
- AUC and KS measure discriminatory power: can the signal separate good borrowers from bad borrowers.
- PSI measures stability: has the applicant population shifted enough that the model or variable may be less reliable.
- WoE and IV measure whether a variable can be transformed into a monotonic, auditable risk signal.
- Together these metrics answer three different questions that matter in production: does the variable rank risk, is the portfolio changing, and can the signal be defended in policy review or audit.

In [ ]:
metric_features = [
    feature for feature in [
        "ext_source_mean",
        "payment_rate",
        "credit_income_ratio",
        "credit_annuity_ratio",
        "bureau_total_debt_ratio",
        "install_late_payment_rate",
    ]
    if feature in master_train.columns
]

metric_rows = []
for feature in metric_features:
    series = master_train[feature]
    filled = series.fillna(series.median())
    auc = roc_auc_score(master_train["TARGET"], filled)
    metric_rows.append(
        {
            "feature": feature,
            "auc": max(auc, 1 - auc),
            "ks": ks_statistic(master_train["TARGET"], filled),
        }
    )

metric_table = pd.DataFrame(metric_rows).sort_values("auc", ascending=False)
metric_table

In [ ]:
psi_features = [feature for feature in ["ext_source_mean", "payment_rate", "credit_income_ratio", "credit_annuity_ratio"] if feature in train.columns and feature in test.columns]
psi_rows = []
for feature in psi_features:
    psi_rows.append(
        {
            "feature": feature,
            "psi_train_vs_test": psi(train[feature], test[feature]),
        }
    )
psi_table = pd.DataFrame(psi_rows).sort_values("psi_train_vs_test", ascending=False)
psi_table

In [ ]:
woe_features = [feature for feature in ["age_years", "payment_rate", "credit_income_ratio", "ext_source_mean"] if feature in master_train.columns]
woe_summary_rows = []
woe_tables = {}
for feature in woe_features:
    table, iv = woe_iv_table(master_train[feature], master_train["TARGET"], bins=5)
    woe_tables[feature] = table
    woe_summary_rows.append({"feature": feature, "iv": iv})

woe_summary = pd.DataFrame(woe_summary_rows).sort_values("iv", ascending=False)
display(woe_summary)
display(woe_tables[woe_summary.iloc[0]["feature"]])

## 4.4.3 Explainable Scorecard Path and Monitoring

This section extends the earlier WoE and PSI discussion into the actual explainable model pipeline used in the project.

Justification:
- The project does not stop at exploratory IV tables. It converts those ideas into a production-style scorecard path.
- Numeric variables are binned into monotonic supervised bins, features are ranked with IV, the final score is estimated with a constrained logistic model, and outputs are translated into points and reason codes.
- Monitoring also becomes more realistic here: PSI is measured on the actual scorecard bins, not just on generic quantile histograms, and the notebook surfaces uncertainty around IV and PSI so the discussion is more defensible in an interview or model review.

In [ ]:
scorecard_metrics_path = OUTPUTS_DIR / "reports" / "scorecard_metrics.json"
scorecard_feature_diag_path = OUTPUTS_DIR / "explainability" / "scorecard_feature_diagnostics.csv"
scorecard_bin_monitoring_path = OUTPUTS_DIR / "explainability" / "scorecard_bin_monitoring.csv"

RUN_SCORECARD_IF_MISSING = False

if RUN_SCORECARD_IF_MISSING and not (
    scorecard_metrics_path.exists()
    and scorecard_feature_diag_path.exists()
    and scorecard_bin_monitoring_path.exists()
):
    from hc_risk.config import PipelineConfig
    from hc_risk.scorecard import run_explainable

    _ = run_explainable(PipelineConfig(root_dir=ROOT), force=False)

scorecard_metrics = json.loads(scorecard_metrics_path.read_text()) if scorecard_metrics_path.exists() else None
scorecard_feature_diagnostics = pd.read_csv(scorecard_feature_diag_path) if scorecard_feature_diag_path.exists() else pd.DataFrame()
scorecard_bin_monitoring = pd.read_csv(scorecard_bin_monitoring_path) if scorecard_bin_monitoring_path.exists() else pd.DataFrame()

{
    "scorecard_metrics_available": scorecard_metrics is not None,
    "feature_diagnostics_shape": tuple(scorecard_feature_diagnostics.shape),
    "bin_monitoring_shape": tuple(scorecard_bin_monitoring.shape),
}

In [ ]:
if scorecard_metrics is None:
    print("Scorecard artifacts not found. Run the explainable stage to populate this section.")
else:
    scorecard_summary = pd.DataFrame(
        [
            {"metric": "ROC AUC", "value": scorecard_metrics.get("roc_auc")},
            {"metric": "KS", "value": scorecard_metrics.get("ks")},
            {"metric": "PR AUC", "value": scorecard_metrics.get("pr_auc")},
            {"metric": "Brier", "value": scorecard_metrics.get("brier")},
            {"metric": "Selected features", "value": len(scorecard_metrics.get("selected_features", []))},
            {"metric": "Monitoring sample train rows", "value": scorecard_metrics.get("monitoring_sample_rows", {}).get("train")},
            {"metric": "Monitoring sample test rows", "value": scorecard_metrics.get("monitoring_sample_rows", {}).get("test")},
        ]
    )
    display(scorecard_summary)

    if not scorecard_feature_diagnostics.empty:
        display(scorecard_feature_diagnostics.sort_values("iv", ascending=False).head(10))
        display(scorecard_feature_diagnostics.sort_values("psi_train_test", ascending=False).head(10))

    if not scorecard_bin_monitoring.empty and not scorecard_feature_diagnostics.empty:
        most_unstable_feature = scorecard_feature_diagnostics.sort_values("psi_train_test", ascending=False).iloc[0]["feature"]
        display(
            scorecard_bin_monitoring[scorecard_bin_monitoring["feature"] == most_unstable_feature]
            .sort_values("psi_component", ascending=False)
        )

In [ ]:
if not scorecard_feature_diagnostics.empty:
    top_iv = scorecard_feature_diagnostics.sort_values("iv", ascending=False).head(8).sort_values("iv")
    top_psi = scorecard_feature_diagnostics.sort_values("psi_train_test", ascending=False).head(8).sort_values("psi_train_test")

    fig, axes = plt.subplots(1, 2, figsize=(14, 6))

    axes[0].barh(top_iv["feature"], top_iv["iv"], color="#4C78A8")
    axes[0].errorbar(
        top_iv["iv"],
        top_iv["feature"],
        xerr=[top_iv["iv"] - top_iv["iv_ci_lower"], top_iv["iv_ci_upper"] - top_iv["iv"]],
        fmt="none",
        ecolor="black",
        capsize=3,
    )
    axes[0].set_title("Top Scorecard Features by IV")
    axes[0].set_xlabel("Information Value")

    axes[1].barh(top_psi["feature"], top_psi["psi_train_test"], color="#E45756")
    axes[1].errorbar(
        top_psi["psi_train_test"],
        top_psi["feature"],
        xerr=[top_psi["psi_train_test"] - top_psi["psi_ci_lower"], top_psi["psi_ci_upper"] - top_psi["psi_train_test"]],
        fmt="none",
        ecolor="black",
        capsize=3,
    )
    axes[1].set_title("Top Scorecard Features by Train/Test PSI")
    axes[1].set_xlabel("Population Stability Index")

    plt.tight_layout()
    plt.show()
else:
    print("Scorecard diagnostics are not available yet.")

### Interview Framing for the Scorecard Section

A concise way to explain this project end to end is:

- I started with domain-first credit analysis to separate capacity, willingness, time-based deterioration, and portfolio drift.
- I then built an explainable scorecard path using WoE-style bins, IV-based feature ranking, a constrained logistic model, and reason-code outputs.
- Instead of stopping at a static score, I added monitoring on the exact scorecard bins with PSI and uncertainty ranges, so the model discussion includes governance and stability, not just discrimination.
- That makes the project easier to explain to both technical interviewers and risk stakeholders because it covers data, model, explanation, and monitoring in one pipeline.

## Borrower Funnel Drilldown by SK_ID_CURR

This section is for end-to-end tracing of one borrower through the project.

Justification:
- Interviewers often want proof that the pipeline is not just aggregate reporting. They want to know whether you can trace one customer from raw application to model outcome.
- A borrower-level drilldown makes the project concrete: raw profile, bureau exposure, prior applications, repayment behavior, selected scorecard features, reason codes, and final model outputs.
- This is also how risk teams typically review edge cases, overrides, and adverse-action narratives.

In [ ]:
scorecard_predictions = pd.read_csv(OUTPUTS_DIR / "predictions" / "scorecard_predictions.csv") if (OUTPUTS_DIR / "predictions" / "scorecard_predictions.csv").exists() else pd.DataFrame()
reason_codes = pd.read_csv(OUTPUTS_DIR / "explainability" / "reason_codes.csv") if (OUTPUTS_DIR / "explainability" / "reason_codes.csv").exists() else pd.DataFrame()
scorecard_bins = pd.read_csv(OUTPUTS_DIR / "explainability" / "scorecard_bins.csv") if (OUTPUTS_DIR / "explainability" / "scorecard_bins.csv").exists() else pd.DataFrame()
leaderboard_oof = pd.read_csv(OUTPUTS_DIR / "predictions" / "leaderboard_oof.csv") if (OUTPUTS_DIR / "predictions" / "leaderboard_oof.csv").exists() else pd.DataFrame()
leaderboard_test = pd.read_csv(OUTPUTS_DIR / "predictions" / "leaderboard_test_predictions.csv") if (OUTPUTS_DIR / "predictions" / "leaderboard_test_predictions.csv").exists() else pd.DataFrame()

CHECK_SK_ID_CURR = 182619

all_applications = pd.concat([application_train.assign(dataset_split="train"), application_test.assign(dataset_split="test")], axis=0, ignore_index=True)
customer_application = all_applications[all_applications["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy()
customer_bureau = bureau[bureau["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy()
customer_previous = previous[previous["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy()
customer_installments = installments[installments["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy()
customer_master = master[master["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy() if master is not None else pd.DataFrame()
customer_scorecard = scorecard_predictions[scorecard_predictions["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy() if not scorecard_predictions.empty else pd.DataFrame()
customer_reasons = reason_codes[reason_codes["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy() if not reason_codes.empty else pd.DataFrame()

customer_split = customer_application["dataset_split"].iloc[0] if not customer_application.empty else None
if customer_split == "train":
    customer_leaderboard = leaderboard_oof[leaderboard_oof["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy() if not leaderboard_oof.empty else pd.DataFrame()
else:
    customer_leaderboard = leaderboard_test[leaderboard_test["SK_ID_CURR"] == CHECK_SK_ID_CURR].copy() if not leaderboard_test.empty else pd.DataFrame()

funnel_summary = pd.DataFrame(
    [
        {"stage": "Application row found", "value": 0 if customer_application.empty else 1},
        {"stage": "Dataset split", "value": customer_split},
        {"stage": "Bureau rows", "value": int(len(customer_bureau))},
        {"stage": "Previous applications", "value": int(len(customer_previous))},
        {"stage": "Installment rows", "value": int(len(customer_installments))},
        {"stage": "Master feature row", "value": 0 if customer_master.empty else 1},
        {"stage": "Scorecard prediction available", "value": 0 if customer_scorecard.empty else 1},
        {"stage": "Leaderboard prediction available", "value": 0 if customer_leaderboard.empty else 1},
    ]
)
funnel_summary

In [ ]:
if customer_application.empty:
    print(f"SK_ID_CURR {CHECK_SK_ID_CURR} not found in application_train/application_test.")
else:
    app_cols = [
        col for col in [
            "SK_ID_CURR",
            "dataset_split",
            "TARGET",
            "NAME_CONTRACT_TYPE",
            "CODE_GENDER",
            "NAME_INCOME_TYPE",
            "NAME_EDUCATION_TYPE",
            "NAME_FAMILY_STATUS",
            "NAME_HOUSING_TYPE",
            "CNT_CHILDREN",
            "CNT_FAM_MEMBERS",
            "AMT_INCOME_TOTAL",
            "AMT_CREDIT",
            "AMT_ANNUITY",
            "AMT_GOODS_PRICE",
            "DAYS_BIRTH",
            "DAYS_EMPLOYED",
            "EXT_SOURCE_1",
            "EXT_SOURCE_2",
            "EXT_SOURCE_3",
        ] if col in customer_application.columns
    ]
    display(customer_application[app_cols].T.rename(columns={customer_application.index[0]: "value"}))

    bureau_summary = pd.DataFrame(
        [
            {"metric": "bureau_rows", "value": int(len(customer_bureau))},
            {"metric": "active_bureau_loans", "value": int((customer_bureau.get("CREDIT_ACTIVE") == "Active").sum()) if not customer_bureau.empty else 0},
            {"metric": "bureau_total_credit_sum", "value": float(customer_bureau.get("AMT_CREDIT_SUM", pd.Series(dtype=float)).sum()) if not customer_bureau.empty else 0.0},
            {"metric": "bureau_total_debt_sum", "value": float(customer_bureau.get("AMT_CREDIT_SUM_DEBT", pd.Series(dtype=float)).sum()) if not customer_bureau.empty else 0.0},
            {"metric": "bureau_total_overdue_sum", "value": float(customer_bureau.get("AMT_CREDIT_SUM_OVERDUE", pd.Series(dtype=float)).sum()) if not customer_bureau.empty else 0.0},
            {"metric": "previous_application_rows", "value": int(len(customer_previous))},
            {"metric": "previous_approved", "value": int((customer_previous.get("NAME_CONTRACT_STATUS") == "Approved").sum()) if not customer_previous.empty else 0},
            {"metric": "previous_refused", "value": int((customer_previous.get("NAME_CONTRACT_STATUS") == "Refused").sum()) if not customer_previous.empty else 0},
            {"metric": "installment_rows", "value": int(len(customer_installments))},
            {"metric": "avg_installment_payment", "value": float(customer_installments.get("AMT_PAYMENT", pd.Series(dtype=float)).mean()) if not customer_installments.empty else np.nan},
        ]
    )
    display(bureau_summary)

    if not customer_previous.empty:
        display(customer_previous.sort_values("DAYS_DECISION", ascending=False).head(10))
    if not customer_installments.empty:
        customer_installments = customer_installments.copy()
        customer_installments["days_past_due"] = (customer_installments["DAYS_ENTRY_PAYMENT"] - customer_installments["DAYS_INSTALMENT"]).clip(lower=0)
        display(customer_installments.sort_values("NUM_INSTALMENT_NUMBER").head(15))

In [ ]:
def parse_interval_label(label):
    match = re.match(r"^([\[(])\s*([^,]+),\s*([^\])]+)([\])])$", str(label))
    if not match:
        return None
    left_bracket, left_raw, right_raw, right_bracket = match.groups()
    left = -np.inf if left_raw.strip() == "-inf" else float(left_raw)
    right = np.inf if right_raw.strip() == "inf" else float(right_raw)
    return {
        "left": left,
        "right": right,
        "left_closed": left_bracket == "[",
        "right_closed": right_bracket == "]",
    }

def resolve_scorecard_bin(value, feature_bins, feature_type):
    if feature_bins.empty:
        return None
    if pd.isna(value):
        hit = feature_bins[feature_bins["label"] == "__MISSING__"]
        return hit.iloc[0] if not hit.empty else None
    if feature_type == "categorical":
        hit = feature_bins[feature_bins["label"] == str(value)]
        if hit.empty:
            hit = feature_bins[feature_bins["label"] == "__OTHER__"]
        return hit.iloc[0] if not hit.empty else None
    for _, row in feature_bins.iterrows():
        interval = parse_interval_label(row["label"])
        if interval is None:
            continue
        left_ok = value >= interval["left"] if interval["left_closed"] else value > interval["left"]
        right_ok = value <= interval["right"] if interval["right_closed"] else value < interval["right"]
        if left_ok and right_ok:
            return row
    return None

if customer_master.empty or customer_scorecard.empty:
    print("Master row or scorecard artifacts missing for this ID.")
else:
    selected_features = scorecard_metrics.get("selected_features", []) if scorecard_metrics is not None else []
    reason_feature_set = set()
    if not customer_reasons.empty:
        for col in ["reason_code_1", "reason_code_2", "reason_code_3"]:
            if col in customer_reasons.columns:
                reason_feature_set.update(customer_reasons[col].dropna().astype(str).tolist())

    feature_rows = []
    customer_values = customer_master.iloc[0]
    for feature in selected_features:
        if feature not in customer_master.columns or scorecard_bins.empty:
            continue
        feature_bins = scorecard_bins[scorecard_bins["feature"] == feature].copy()
        if feature_bins.empty:
            continue
        feature_type = feature_bins["feature_type"].iloc[0]
        value = customer_values[feature]
        matched = resolve_scorecard_bin(value, feature_bins, feature_type)
        feature_rows.append(
            {
                "feature": feature,
                "raw_value": value,
                "matched_label": None if matched is None else matched["label"],
                "woe": None if matched is None else matched["woe"],
                "points": None if matched is None else matched["points"],
                "bad_rate": None if matched is None else matched["bad_rate"],
                "is_top_reason": feature in reason_feature_set,
            }
        )

    customer_feature_funnel = pd.DataFrame(feature_rows)
    display(customer_scorecard)
    if not customer_leaderboard.empty:
        display(customer_leaderboard)
    if not customer_reasons.empty:
        display(customer_reasons)
    display(customer_feature_funnel.sort_values(["is_top_reason", "points"], ascending=[False, True]).head(20))

### Interview Framing for the Borrower Funnel

If someone asks how you would inspect one borrower end to end, a good answer is:

- I can start from `SK_ID_CURR` and pull the raw application row, bureau exposure, prior applications, and installment history.
- Then I map that borrower into the engineered feature space and show which explainable scorecard features were active.
- After that I can show the exact scorecard output, the top reason codes, and any leaderboard-style model prediction that was produced.
- That proves the project is not just a leaderboard exercise; it supports borrower-level diagnostics and model governance.

## 4.4.4 Special Values, Missing Data, and Outliers

Credit datasets are not clean by default, and those irregularities often carry real business meaning.

Justification:
- Sentinel values, missing bureau fields, and extreme amounts can be informative rather than accidental.
- If they are handled casually, the model can learn data-collection artifacts instead of economic risk.
- In a regulated lending setting, these treatments must be documented because they affect fairness, score stability, and adverse-action explanations.

In [ ]:
special_value_summary = pd.DataFrame(
    [
        {"item": "DAYS_EMPLOYED == 365243 sentinel", "value": int((application_train["DAYS_EMPLOYED"] == 365243).sum())},
        {"item": "EXT_SOURCE_1 missing share", "value": float(application_train["EXT_SOURCE_1"].isna().mean())},
        {"item": "EXT_SOURCE_2 missing share", "value": float(application_train["EXT_SOURCE_2"].isna().mean())},
        {"item": "EXT_SOURCE_3 missing share", "value": float(application_train["EXT_SOURCE_3"].isna().mean())},
        {"item": "AMT_INCOME_TOTAL p99", "value": float(application_train["AMT_INCOME_TOTAL"].quantile(0.99))},
        {"item": "AMT_CREDIT p99", "value": float(application_train["AMT_CREDIT"].quantile(0.99))},
    ]
)
special_value_summary

In [ ]:
missing_top = (
    application_train.isna()
    .mean()
    .sort_values(ascending=False)
    .head(20)
    .rename("missing_fraction")
    .reset_index()
    .rename(columns={"index": "feature"})
)
missing_top

## 4.5 Why AI Matters Here

This section explains where AI adds value and where domain controls still dominate.

Justification:
- AI is useful because the risk evidence is heterogeneous, multi-table, and time-dependent.
- AI is risky because credit decisions require transparency, stable monitoring, and defensible explanations.
- The practical answer is a hybrid approach: domain-led features and monitoring first, flexible models second. That is the logic behind the notebook structure.

In [ ]:
ai_takeaways = pd.DataFrame(
    [
        {
            "theme": "Capacity and willingness are different signals",
            "home_credit_evidence": "repayment ratios and income variables behave differently from bureau / historical delinquency variables",
            "ai_implication": "separate features or sub-models can improve risk clarity",
        },
        {
            "theme": "Default is a trajectory, not a single event",
            "home_credit_evidence": "installment data supports vintage and survival-style monitoring",
            "ai_implication": "time-aware models and early-warning systems are more faithful than one-shot classification alone",
        },
        {
            "theme": "Interpretability still matters",
            "home_credit_evidence": "WoE / IV, risk ladders, and stable feature bins remain intuitive and auditable",
            "ai_implication": "black-box models should sit on top of transparent domain features, not replace them blindly",
        },
        {
            "theme": "Monitoring is part of the model",
            "home_credit_evidence": "PSI, vintage drift, and special-value spikes can all degrade credit decisions",
            "ai_implication": "production credit AI needs continuous data-quality and stability checks",
        },
    ]
)
ai_takeaways

## Presentation Justification

If you need to explain why this notebook is organized this way, use the following logic:

- The notebook is built around credit-risk questions, not around raw CSV files.
- Each block corresponds to a lending control: origination assessment, affordability, behavioral willingness, time-to-distress, population stability, and explainability.
- That structure makes the output easier to connect to real decisions such as approval, pricing, limit management, and collections prioritization.
- The purpose is not to claim causal truth from one dataset. The purpose is to show a defendable analytical framework that a bank, fintech, or regulator-facing risk team would recognize.
- When presenting results, describe the charts as portfolio diagnostics and candidate model signals, not as standalone policy rules.

## Final Analyst Notes

The summary at the end should be read as an analyst brief, not as a final production credit policy. It consolidates the strongest directional findings from the notebook so they can be reviewed, challenged, and translated into model design or policy decisions.

In [ ]:
final_notes = {
    "portfolio_default_rate": float(train["TARGET"].mean()),
    "strongest_univariate_metrics": metric_table.head(5).to_dict(orient="records"),
    "highest_psi_features": psi_table.head(5).to_dict(orient="records"),
    "top_iv_features": woe_summary.head(5).to_dict(orient="records"),
    "key_data_quality_flags": special_value_summary.to_dict(orient="records"),
    "scorecard_summary": (
        {
            "roc_auc": scorecard_metrics.get("roc_auc"),
            "ks": scorecard_metrics.get("ks"),
            "selected_features": scorecard_metrics.get("selected_features", [])[:10],
            "top_iv_diagnostics": scorecard_metrics.get("feature_diagnostics_top_iv", []),
            "top_psi_diagnostics": scorecard_metrics.get("feature_diagnostics_top_psi", []),
        }
        if scorecard_metrics is not None
        else None
    ),
    "borrower_drilldown": (
        {
            "sk_id_curr": CHECK_SK_ID_CURR,
            "dataset_split": customer_split,
            "scorecard_probability": None if customer_scorecard.empty else float(customer_scorecard.iloc[0]["scorecard_probability"]),
            "scorecard_score": None if customer_scorecard.empty else float(customer_scorecard.iloc[0]["scorecard_score"]),
            "top_reason_codes": [] if customer_reasons.empty else [
                customer_reasons.iloc[0].get("reason_code_1"),
                customer_reasons.iloc[0].get("reason_code_2"),
                customer_reasons.iloc[0].get("reason_code_3"),
            ],
        }
        if not customer_application.empty
        else None
    ),
}
final_notes